## INIT

In [0]:
import sys
import os
sys.path.append(os.path.abspath('../../'))
from utils.logger_utils import get_project_logger

logger = get_project_logger("Gold_Dimension_vehicle_type")
logger.info("--- Starting Gold Layer: Creating dim_manufacturer ---")

## Creating a gold dimension view

In [0]:
try:
    logger.info("Step 1/2: Running SQL to create dim_vehicle_type")
    
    sql_query = """
    CREATE OR REPLACE VIEW transport_lakehouse.gold.dim_vehicle_type AS
    SELECT
    ROW_NUMBER() OVER (ORDER BY vehicle_type_cd, vehicle_type_nm) AS vehicle_type_key,
    vehicle_type_cd,
    vehicle_type_nm,
    MAX(vehicle_type_eu_cd) AS vehicle_type_eu_cd
    FROM (
    SELECT
        CAST(vehicle_type_cd AS STRING) AS vehicle_type_cd,
        vehicle_type_nm,
        CAST(vehicle_type_eu_cd AS STRING) AS vehicle_type_eu_cd
    FROM transport_lakehouse.silver.silver_vehicles_public

    UNION ALL

    SELECT
        CAST(vehicle_type_cd AS STRING) AS vehicle_type_cd,
        vehicle_type_nm,
        NULL AS vehicle_type_eu_cd
    FROM transport_lakehouse.silver.silver_vehicles_two_wheeled
    ) t
    WHERE vehicle_type_cd IS NOT NULL AND vehicle_type_nm IS NOT NULL
    GROUP BY vehicle_type_cd, vehicle_type_nm
    """
    
    spark.sql(sql_query)
    logger.info("Step 1/2: View created successfully.")

    logger.info("Step 2/2: Performing Quality Check (Row Count)")
    dim_count = spark.table("transport_lakehouse.gold.dim_vehicle_type").count()
    logger.info(f"Quality Check Passed: dim_vehicle_type contains {dim_count} unique vehicle types.")

except Exception as e:
    logger.error(f"Failed to create Gold Dimension: {str(e)}")
    raise e

logger.info("--- Gold dim_vehicle_type Process Completed ---")
